# heart-atlas-reference-mapping — Google Colab (GPU)

Runs the **main** donor-held-out experiment end to end on a free GPU runtime.
The sealed query labels are only opened in the dedicated evaluation section,
after both prediction files are written.

## Before you start
1. **Runtime → Change runtime type → T4 GPU.**
2. Set `REPO_URL` below to your GitHub fork/clone URL (the repo must be pushed
   there; Colab clones it).
3. Run cells top to bottom. Do not re-run training cells after evaluation.

Free Colab is an interactive service, not an automation backend. Keep the
browser tab open and respond to the captcha if prompted.

## 1. Verify the GPU

Expected: a CUDA device (Tesla T4). If `cuda_available` is False, use Runtime → Factory reset and reselect T4.

In [ ]:
import torch, sys
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Enable the T4 GPU runtime before continuing"


## 2. Install pinned dependencies

scvi-tools 1.3.3 is the last line supporting the analysis as written (needs Python 3.10/3.11). After installing, the runtime may ask you to restart — do so once, then continue from the next cell (installs persist for the session).

In [ ]:
%pip install -q \
  "scvi-tools==1.3.3" "scanpy==1.11.5" "anndata==0.11.4" \
  "numpy==1.26.4" "numba==0.60.0" "llvmlite==0.43.0" \
  "scikit-misc==0.5.2" "igraph==1.0.0" "leidenalg==0.12.0" \
  "torch==2.2.2" "lightning==2.6.6"
print("install complete; restart the runtime ONLY if Colab prompts you")


## 3. Clone the repository

Replace the placeholder with your fork URL. The working directory becomes the repo root for every later cell.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/<YOUR_USER>/heart-atlas-reference-mapping.git"
REPO_DIR = "/content/heart-atlas-reference-mapping"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
%cd /content/heart-atlas-reference-mapping
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
!git rev-parse --short HEAD


## 4. Download the 20k heart atlas

The official scvi-tools loader URL is used; file ~66 MB. The audit cell below confirms raw integer counts before any training.

In [ ]:
import sys; sys.path.insert(0, "src")
from heartmap.config import load_config
from heartmap.data import load_heart_dataset, validate_counts
cfg = load_config("configs/main.yaml")
adata = load_heart_dataset(str(cfg.data_raw_dir), remove_nuisance_clusters=True)
adata.layers[cfg["counts_layer"]] = adata.X.copy()
print(validate_counts(adata, cfg["counts_layer"]))


## 5. Fix the donor-held-out split and seal D6 labels

Query donor D6 is fixed deterministically. This cell writes the sealed label file that training never reads.

In [ ]:
!python scripts/prepare_split.py --config configs/main.yaml


## 6. Run the PCA + kNN baseline (predictions only)

In [ ]:
!python scripts/run_baseline.py --config configs/main.yaml


## 7. Train scVI + scANVI on the 13 reference donors

A few minutes on a T4. Watch for decreasing `train_loss_epoch`; early stopping ends scVI automatically. Do not re-run this cell later.

In [ ]:
!python scripts/train_reference.py --config configs/main.yaml


## 8. scArches update onto D6 and soft predictions

Reference weights are frozen; the log prints trainable vs total parameters. Sealed labels are not accessed.

In [ ]:
!python scripts/map_query.py --config configs/main.yaml


## 9. Freeze point — then evaluate

Both prediction files now exist. This is the **only** stage that opens `query_eval_labels_main.csv`.

In [ ]:
!python scripts/evaluate.py --config configs/main.yaml


## 10. Generate figures

In [ ]:
!python scripts/make_figures.py --config configs/main.yaml


## 11. Run the full verification suite

pytest (CPU) plus the artifact verifier with the strict main-run flag.

In [ ]:
!python -m pytest tests/ -q
!python scripts/verify_outputs.py --config configs/main.yaml --require-complete-main-run


## 12. Package results and download

Models and latents are large; the zip contains predictions, metrics, figures, manifests, training summaries and model weights.

In [ ]:
import zipfile, os
from google.colab import files
zip_path = "/content/heart_atlas_main_run.zip"
include_dirs = ["results", "figures", "data/splits", "data/processed", "models"]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for d in include_dirs:
        for root, _, fnames in os.walk(d):
            for f in fnames:
                p = os.path.join(root, f)
                if f.endswith((".h5ad", ".pt", ".csv", ".json", ".png", ".txt")):
                    z.write(p)
print("zip size MB:", round(os.path.getsize(zip_path) / 1e6, 1))
files.download(zip_path)


## 13. (Optional) Save to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp /content/heart_atlas_main_run.zip "/content/drive/MyDrive/"


## After downloading
Unzip into your local clone (replacing placeholder artifacts), rerun `python scripts/verify_outputs.py --require-complete-main-run` locally, then commit code/configs/CSVs/figures per `.gitignore`. See `docs/COLAB_RUNBOOK.md`.